# 혜진 v5: 아미노산 변화·학습 지원도·직접 특성 학습
노트북 하나에서 파생변수 생성, 학습 내 선별/튜닝, v4 대조 실험, 재분할 점검, 홀드아웃 재평가, 전체 학습, 피클·제출·보고서 생성을 수행합니다. 다른 프로젝트 Python 파일이나 이전 모델 피클이 필요하지 않습니다.

기존 실제 Public은 v1 0.2765425197 → v3 0.2998780222 → **v4 0.3169555443**입니다. v5 Public은 제출 전 알 수 없습니다. 기존 홀드아웃을 반복 관측했고 개발 데이터에도 여러 번 적응했으므로 새로운 독립 평가가 아닙니다. 테스트의 빈도/예측 비율로 학습하거나 강제 비율 정렬을 하지 않습니다.

가설: ① 위치 번호가 달라도 같은 아미노산 변화는 보완 단서가 될 수 있다. ② 학습에서 반복 관측된 변이와 낯선 변이를 같은 강도로 보지 않는 표현이 도움이 될 수 있다. ③ 토큰 수 대신 유전자별 요약을 쓰면 중복 표기에 덜 민감할 수 있다. 가설이지 병원성 판정이나 대회 성능 보장이 아닙니다.

필요 패키지: numpy, pandas, scipy, scikit-learn(현재 1.9), lightgbm, xgboost, matplotlib, cloudpickle, threadpoolctl, IPython. 공용 data/의 CSV 세 개가 필요합니다. 결과 artifacts/v5/, 새 제출 submission/혜진_submission_v5.csv. 기존 제출은 덮어쓰지 않습니다.

실행 기록: 초기 44개 설정에 개발 CV를 보고 규제·지원도 12개와 희귀 변이 증강 3개를 추가한 총 59개 설정입니다. 같은 개발 데이터에 반복 적응하므로 선택 편향이 있습니다. 아래 Run All 코드는 59개를 처음부터 평가합니다. 기존 시험 결과의 수치를 새 모델 점수로 대신 쓰지 않습니다.

검증 환경은 Python 3.15와 requirements-v5.txt에 기록한 버전입니다. macOS에서 LightGBM/XGBoost 로딩에 실패하면 OpenMP 런타임 libomp가 필요할 수 있습니다. 최종 앙상블에는 v4가 일부 포함되므로 신규 표현의 중복 불변성이 전체 앙상블의 완전한 불변성을 뜻하지는 않습니다.

In [1]:
import os,sys,re,json,pickle,cloudpickle,hashlib,platform,time,copy,warnings
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
from scipy import sparse
from scipy.special import softmax
from sklearn.base import BaseEstimator,TransformerMixin,clone
from sklearn.feature_extraction import DictVectorizer
from sklearn.feature_selection import chi2,f_classif
from sklearn.preprocessing import StandardScaler,MaxAbsScaler
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import f1_score,accuracy_score,classification_report,confusion_matrix
from lightgbm import LGBMClassifier
from threadpoolctl import threadpool_limits
from IPython.display import display,FileLink
threadpool_limits(2)
SEED=42
roots=[Path.cwd(),*Path.cwd().parents]
REPO=next((p for p in roots if all((p/'data'/f).is_file() for f in ['train.csv','test.csv','sample_submission.csv'])),None)
if REPO is None:raise FileNotFoundError('공用 data/에 train.csv, test.csv, sample_submission.csv가 필요합니다.')
DATA=REPO/'data';WORK=REPO/'models'/'혜진' if (REPO/'models'/'혜진').is_dir() else REPO
OUT=WORK/'artifacts'/'v5';OUT.mkdir(parents=True,exist_ok=True)
SUBMIT=REPO/'submission';SUBMIT.mkdir(exist_ok=True)
os.environ['MPLCONFIGDIR']=str(OUT/'.mplconfig')
train=pd.read_csv(DATA/'train.csv');test=pd.read_csv(DATA/'test.csv');sample=pd.read_csv(DATA/'sample_submission.csv')
genes=[c for c in train if c not in ['ID','SUBCLASS']]
assert list(test)==['ID']+genes and sample.ID.equals(test.ID)
assert train.ID.is_unique and test.ID.is_unique
y=train.SUBCLASS.to_numpy();labels=np.sort(np.unique(y));yi=np.searchsorted(labels,y);NC=len(labels)
X=train[genes];groups=pd.util.hash_pandas_object(X,index=False).to_numpy()
dev,hold=next(StratifiedGroupKFold(5,shuffle=True,random_state=SEED).split(X,y,groups))
folds=list(StratifiedGroupKFold(3,shuffle=True,random_state=SEED+1).split(X.iloc[dev],y[dev],groups[dev]))
assert not set(groups[dev]).intersection(groups[hold])
for a,b in folds:assert not set(groups[dev[a]]).intersection(groups[dev[b]])
pd.DataFrame({'ID':train.ID,'split':np.where(np.isin(np.arange(len(train)),hold),'holdout','development'),'group':groups.astype(str)}).to_csv(OUT/'split_manifest.csv',index=False)
print('DATA',DATA,'OUTPUT',OUT,'Train/test',train.shape,test.shape,'Dev/holdout',len(dev),len(hold))

from sklearn.neural_network import MLPClassifier
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier


DATA /Users/mac/project/cancer_project/data OUTPUT /Users/mac/project/cancer_project/artifacts/v5 Train/test (6201, 4386) (2546, 4385) Dev/holdout 4960 1241


## 1. v4 파서와 v5 행별 표현
기존 v4의 X→* 파서와 정확한 기준 모델을 보존합니다. v5는 p. 접두사·종결·프레임시프트 표기를 정규화하고, 한 유전자 셀 안의 동일 토큰을 중복 제거합니다. 위치를 무조건 ±몇 칸으로 맞추지 않습니다. 대신 `IDH1:R>H`처럼 유전자+아미노산 변화 단서를 추가합니다. 동일 변화가 같은 생물학적 효과라는 보장은 없습니다.

In [2]:
GROUPS = {
 'p53_cellcycle': 'TP53 RB1 CDKN2A CDKN2B CDK4 CDK6 CCND1 ATM CHEK2'.split(),
 'rtk_ras_mapk': 'EGFR ERBB2 ERBB3 KRAS NRAS HRAS BRAF NF1 MAP2K1 MAP2K2 KIT PDGFRA FGFR2 FGFR3'.split(),
 'pi3k_mtor': 'PIK3CA PIK3R1 PTEN AKT1 MTOR TSC1 TSC2 STK11'.split(),
 'dna_repair': 'BRCA1 BRCA2 PALB2 ATM ATR CHEK2 MLH1 MSH2 MSH6 PMS2 POLE POLD1'.split(),
 'chromatin': 'ARID1A ARID1B ARID2 PBRM1 SMARCA4 BAP1 SETD2 KMT2A KMT2C KMT2D ATRX'.split(),
 'wnt': 'APC CTNNB1 AXIN1 AXIN2 RNF43'.split(),
 'metabolism': 'IDH1 IDH2 SDHB SDHC SDHD FH VHL'.split(),
}

def kind(token):
 if 'fs' in token.lower(): return 'frameshift'
 if '*' in token: return 'stop'
 if 'del' in token.lower(): return 'deletion'
 if 'ins' in token.lower(): return 'insertion'
 m = re.fullmatch(r'([A-Z])(\d+)([A-Z])', token)
 if m: return 'synonymous' if m[1] == m[3] else 'substitution'
 return 'other'

class MutationFeatures(BaseEstimator, TransformerMixin):
 def __init__(self, mode='full', min_df=3):
  self.mode = mode
  self.min_df = min_df

 def dictionaries(self, X):
  genes = list(X.columns)
  for row in X.itertuples(index=False, name=None):
   d = {}; counts = {}; burden = 0; events = 0; positions = []
   changed = set()
   for gene, value in zip(genes, row):
    if value is None or str(value) in ('nan', '', '<NA>'):
     d['missing:' + gene] = 1; continue
    if value == 'WT': continue
    changed.add(gene); burden += 1; d['gene:' + gene] = 1
    if self.mode == 'binary': continue
    tokens = str(value).split()
    for raw_token in tokens:
     token = re.sub(r'^([A-Z]\d+)X$', r'\1*', raw_token)
     events += 1; typ = kind(token); counts[typ] = counts.get(typ, 0) + 1
     d['variant:' + gene + ':' + token] = 1
     d['type:' + gene + ':' + typ] = 1
     m = re.search(r'\d+', token)
     if m:
      position = int(m[0]); positions.append(position)
      d['position:' + gene + ':' + str(position)] = 1
   if self.mode == 'full':
    d['numeric:log_mutated_genes'] = np.log1p(burden)
    d['numeric:log_events'] = np.log1p(events)
    d['numeric:multi_event_excess'] = np.log1p(max(0, events-burden))
    for typ in ['frameshift','stop','deletion','insertion','synonymous','substitution','other']:
     d['numeric:log_' + typ] = np.log1p(counts.get(typ,0))
     d['numeric:fraction_' + typ] = counts.get(typ,0)/max(events,1)
    d['numeric:log_mean_position'] = np.log1p(np.mean(positions)) if positions else 0.
    for name, members in GROUPS.items():
     d['module:' + name] = np.log1p(len(changed.intersection(members)))
   yield d

 def fit(self, X, y=None):
  self.genes_ = list(X.columns)
  self.vectorizer_ = DictVectorizer(dtype=np.float32)
  matrix = self.vectorizer_.fit_transform(self.dictionaries(X))
  names = self.vectorizer_.get_feature_names_out()
  self.keep_ = (np.asarray((matrix != 0).sum(axis=0)).ravel() >= self.min_df) | np.array([s.startswith(('numeric:', 'module:')) for s in names])
  self.feature_names_ = names[self.keep_]
  return self

 def transform(self, X):
  missing = set(self.genes_) - set(X.columns)
  if missing: raise ValueError('Missing gene columns: ' + str(sorted(missing)[:10]))
  return sparse.csr_matrix(self.vectorizer_.transform(self.dictionaries(X[self.genes_]))[:, self.keep_])

 def get_feature_names_out(self, input_features=None): return self.feature_names_
def canon(t):
 t=t.strip().removeprefix('p.');t=re.sub(r'(?<=\d)X$','*',t)
 return re.sub(r'^([A-Z]\d+)(?:[A-Z*]+)?fs.*$',r'\1fs',t)
def aa_change(t):
 m=re.fullmatch(r'([A-Z])\d+([A-Z*])',t)
 if m:return m[1]+'>'+m[2]
 m=re.match(r'([A-Z])\d+',t)
 if m:
  for suffix in ['fs','del','ins']:
   if suffix in t:return m[1]+'>'+suffix
 return None

def parse_events(frame):
 names=list(frame.columns);result=[]
 for row in frame.itertuples(index=False,name=None):
  result.append([(g,tuple(sorted(set(canon(t) for t in v.split())))) for g,v in zip(names,row) if isinstance(v,str) and v!='WT'])
 return result

class EvidenceEncoder:
 def __init__(self,unknown=1.,silent=1.,support=2):self.unknown=unknown;self.silent=silent;self.support=support
 def fit(self,events):
  self.token_df={};self.change_df={}
  if self.unknown<1:
   tokens=Counter();changes=Counter()
   for row in events:
    for g,ts in row:
     tokens.update((g,t) for t in ts)
     changes.update(set((g,a) for t in ts if (a:=aa_change(t)) is not None))
   self.token_df={k:v for k,v in tokens.items() if v>=self.support}
   self.change_df={k:v for k,v in changes.items() if v>=self.support}
  return self
 def transform(self,events,training=False,exclude_events=None):
  result=[]
  for row_number,row in enumerate(events):
   own=exclude_events[row_number] if exclude_events is not None else (row if training else [])
   own_tokens={(gg,tt) for gg,tts in own for tt in tts}
   own_changes={(gg,ac) for gg,tts in own for tt in tts if (ac:=aa_change(tt)) is not None}
   d={};gene_weights={};type_sums=Counter();non_syn=0.;trunc=0.;multikind=0
   for g,ts in row:
    types={};gene=0.;known_gene=False
    for t in ts:
     typ=kind(t);a=aa_change(t)
     known=self.token_df.get((g,t),0)-int((g,t) in own_tokens)>=self.support
     known_aa=a is not None and self.change_df.get((g,a),0)-int((g,a) in own_changes)>=self.support
     if typ=='synonymous':w=self.silent
     elif known or typ in ['stop','frameshift','deletion','insertion']:w=1.
     elif known_aa:w=max(self.unknown,.75)
     else:w=self.unknown
     if w<=0:continue
     gene=max(gene,w);types[typ]=max(types.get(typ,0),w)
     d['variant:'+g+':'+t]=max(d.get('variant:'+g+':'+t,0),w)
     m=re.search(r'\d+',t)
     if m:d['position:'+g+':'+m[0]]=max(d.get('position:'+g+':'+m[0],0),w)
     if a:d['aa:'+g+':'+a]=max(d.get('aa:'+g+':'+a,0),w)
     if known and typ!='synonymous':known_gene=True
    if gene<=0:continue
    d['gene:'+g]=gene;gene_weights[g]=gene
    for typ,w in types.items():d['type:'+g+':'+typ]=w;type_sums[typ]+=w
    if known_gene:d['supported:'+g]=1.
    non_syn+=max([w for k,w in types.items() if k!='synonymous'] or [0.])
    trunc+=max([w for k,w in types.items() if k in ['stop','frameshift','deletion','insertion']] or [0.])
    multikind+=len(types)>1
   n=len(row)
   d['numeric:log_raw_genes']=np.log1p(n)
   d['numeric:log_effective_genes']=np.log1p(sum(gene_weights.values()))
   d['numeric:log_nonsyn_genes']=np.log1p(non_syn)
   d['numeric:log_trunc_genes']=np.log1p(trunc)
   d['numeric:log_multikind_genes']=np.log1p(multikind)
   for typ in ['synonymous','substitution','stop','frameshift','deletion','insertion','other']:
    d['numeric:gene_fraction_'+typ]=type_sums[typ]/max(n,1)
   for name,gs in GROUPS.items():d['module:'+name]=np.log1p(sum(gene_weights.get(g,0) for g in gs))
   result.append(d)
  return result

events=parse_events(X)
legacy_records=list(MutationFeatures(mode='full').dictionaries(X))
assert canon('R213X')=='R213*' and canon('K16Rfs')=='K16fs'
assert aa_change('R132H')==aa_change('R140H')=='R>H'
print('Parsed rows:',len(events),'Normalization examples passed')


Parsed rows: 6201 Normalization examples passed


## 2. 학습 내 변수 선택·입력 변환
범주 사전/min_df=3, chi2/ANOVA, Pearson 중복 제거는 학습 부분에서만 적합합니다. 선택 변수는 주로 1,024/2,048개, 상관 임계값 0.98입니다. v4 대조군은 기존 0.95와 같은 설정입니다.

학습 지원도 사전은 정답 라벨을 쓰지 않습니다. 학습 행은 자기 출현을 1 빼고 다른 환자 2명 이상에서 관측됐는지 판단합니다. 검증·테스트는 학습 사전을 조회만 합니다. 낯선 치환의 가중치 0.25, 동의적 변이의 가중치 0/0.25/1은 비교 가설이며 병원성 판정이 아닙니다.

지원도 임계값 2/3과 신경망 alpha 0.01/0.1/1/10을 비교합니다. 추가 증강 3개 설정은 해당 학습에서 관측된 희귀 변이 10개를 붙인 학습 복사본에 0.5 가중치를 줍니다. 사전/변수 선택은 실제 학습 행만으로 적합하고, 증강 행의 원래 환자에 대해서만 자기 출현을 제외합니다. 합성 변이가 라벨을 유지한다는 가정은 현실성 한계가 있습니다.

In [3]:
def relevance_scores(a,target,method):
 base=method.removesuffix('_full')
 if base=='embedded':
  scaled=csr32(MaxAbsScaler().fit_transform(a))
  fitted=OneVsRestClassifier(LogisticRegression(solver='liblinear',l1_ratio=1,C=.2,class_weight='balanced',max_iter=2000),n_jobs=1).fit(scaled,target)
  return np.max(np.abs(np.vstack([e.coef_[0] for e in fitted.estimators_])),axis=0)
 return (chi2(a,target) if base=='chi2' else f_classif(a,target))[0]

def csr32(a):
 b=sparse.csr_matrix(a,dtype=np.float32)
 if b.nnz>=np.iinfo(np.int32).max:raise ValueError('Sparse matrix too large for tree index type')
 b.indices=b.indices.astype(np.int32,copy=False);b.indptr=b.indptr.astype(np.int32,copy=False)
 return b

class Space:
 def fit(self,rs):
  self.vec=DictVectorizer(dtype=np.float32);a=self.vec.fit_transform(rs)
  names=self.vec.get_feature_names_out();df=np.asarray((a!=0).sum(axis=0)).ravel()
  keep=(df>=3)|np.array([n.startswith(('numeric:','module:')) for n in names])
  self.raw=np.flatnonzero(keep);self.names=names[keep]
  return csr32(a[:,self.raw])
 def transform(self,rs):return csr32(self.vec.transform(rs)[:,self.raw])
 def compact_vec(self,indices):
  v=copy.deepcopy(self.vec);mask=np.zeros(len(v.feature_names_),bool);mask[self.raw[np.sort(indices)]]=True
  v.restrict(mask);return v

class Selector:
 def __init__(self,a,target,names):
  self.a=a;self.y=target;self.names=names;self.cache={};self.corr={};self.scores={};self.audit=[];self.examples=[]
 def order(self,method,threshold):
  key=(method,threshold)
  if key in self.cache:return self.cache[key]
  if method not in self.scores:
   with warnings.catch_warnings():
    warnings.simplefilter('ignore',RuntimeWarning);score=relevance_scores(self.a,self.y,method)
   score=np.nan_to_num(score,nan=0,posinf=np.finfo(np.float32).max,neginf=0)
   if not method.endswith('_full'):score[np.array([n.startswith('numeric:') for n in self.names])]=0
   self.scores[method]=score
  score=self.scores[method]
  if method not in self.corr:
   pool=np.argsort(-score,kind='stable')[:min(4096,int((score>0).sum()))]
   z=self.a[:,pool];mean=np.asarray(z.mean(axis=0)).ravel();var=np.asarray(z.multiply(z).mean(axis=0)).ravel()-mean**2
   sd=np.sqrt(np.maximum(var,1e-12));corr=(z.T@z).toarray()/z.shape[0]-np.outer(mean,mean)
   corr/=np.outer(sd,sd);np.clip(corr,-1,1,out=corr)
   self.corr[method]=(pool,corr)
  pool,corr=self.corr[method];chosen=[]
  for i in range(len(pool)):
   if chosen:
    correlations=np.abs(corr[i,chosen]);j=int(np.argmax(correlations))
    if correlations[j]>=threshold:
     if len(self.examples)<100:self.examples.append({'method':method,'threshold':threshold,'kept':self.names[pool[chosen[j]]],'discarded':self.names[pool[i]],'correlation':float(corr[i,chosen[j]])})
     continue
   chosen.append(i)
   if len(chosen)>=2048:break
  order=pool[chosen];self.cache[key]=order
  self.audit.append({'method':method,'threshold':threshold,'association_pool':len(pool),'remaining_up_to_2048':len(order)})
  return order
 def select(self,method,threshold,k):return np.sort(self.order(method,threshold)[:k])

class CompactPredictor:
 def __init__(self,space,cols,est):
  self.vec=space.compact_vec(cols);self.names=self.vec.get_feature_names_out();self.est=est
  self.direct_inputs=len(self.names);self.raw_inputs=len(self.names);self.kind='compact'
 def predict(self,rs):return self.est.predict_proba(csr32(self.vec.transform(rs)))

def background_frame(frame,training_frame,seed,count):
 rng=np.random.default_rng(seed);out=frame.copy();arr=out.to_numpy(copy=True)
 rate=(training_frame.notna()&training_frame.ne('WT')).mean().to_numpy();prob=rate/rate.sum();idx=np.flatnonzero(prob>0)
 for r in range(len(arr)):
  n=min(len(idx),int(rng.poisson(count)))
  for c in rng.choice(len(prob),size=n,replace=False,p=prob):
   token='A1A' if rng.random()<.5 else 'A1V'
   arr[r,c]=token if not isinstance(arr[r,c],str) or arr[r,c]=='WT' else arr[r,c]+' '+token
 return pd.DataFrame(arr,columns=frame.columns,index=frame.index)
class ColumnPost:
 def __init__(self,mode='none'):self.mode=mode
 def fit(self,a,names):
  self.raw_names=np.array(names);self.names=np.array(names)
  if self.mode=='maxabs':self.scaler=MaxAbsScaler().fit(a)
  if self.mode=='block':
   self.binary=np.flatnonzero([not n.startswith(('numeric:','module:')) for n in names])
   self.summary=np.flatnonzero([n.startswith(('numeric:','module:')) for n in names])
   self.tf=TfidfTransformer(norm='l2',use_idf=True).fit(a[:,self.binary])
   self.scaler=MaxAbsScaler().fit(a[:,self.summary])
   self.names=np.concatenate([self.raw_names[self.binary],self.raw_names[self.summary]])
  return self
 def transform(self,a):
  if self.mode=='maxabs':return csr32(self.scaler.transform(a))
  if self.mode=='block':return csr32(sparse.hstack([self.tf.transform(a[:,self.binary]),self.scaler.transform(a[:,self.summary])],format='csr'))
  return csr32(a)

class Member:
 def __init__(self,name,view,encoder,space,cols,post,est):
  self.name=name;self.view=view;self.encoder=encoder;self.vec=space.compact_vec(cols);self.post=post;self.est=est
  self.raw_names=self.vec.get_feature_names_out();self.names=post.names
 def predict_rows(self,rs):return self.est.predict_proba(self.post.transform(csr32(self.vec.transform(rs))))

VIEWS={'raw':(1.,1.),'syn_down':(1.,.25),'gated':(.25,1.),'gated_syn_down':(.25,.25),'gated_nonsyn':(.25,0.)}
CONFIGS={}
for view in VIEWS:
 for k in [1024,2048]:
  for c in [.1,.5]:CONFIGS[f'{view}_lr_k{k}_C{c}']=dict(view=view,algo='lr',k=k,C=c,post='none',method='chi2_full')
  CONFIGS[f'{view}_lgb_k{k}']=dict(view=view,algo='lgb',k=k,post='none',method='f_classif_full')
for view in ['raw','gated_syn_down']:
 for depth in [3,5]:CONFIGS[f'{view}_xgb_d{depth}']=dict(view=view,algo='xgb',k=2048,depth=depth,post='none',method='f_classif_full')
 for alpha in [1,10]:CONFIGS[f'{view}_mlp_a{alpha}']=dict(view=view,algo='mlp',k=1024,alpha=alpha,post='maxabs',method='chi2_full')
 for c in [1,4]:CONFIGS[f'{view}_blocklr_C{c}']=dict(view=view,algo='lr',k=2048,C=c,post='block',method='chi2_full')
 CONFIGS[f'{view}_extra']=dict(view=view,algo='extra',k=1024,post='none',method='f_classif_full')
(OUT/'configurations.json').write_text(json.dumps(CONFIGS,indent=2))

def estimator(cfg):
 algo=cfg['algo']
 if algo=='lr':return LogisticRegression(C=cfg['C'],class_weight='balanced',max_iter=2000)
 if algo=='lgb':return LGBMClassifier(n_estimators=300,learning_rate=.04,num_leaves=15,min_child_samples=20,reg_lambda=10,colsample_bytree=.85,class_weight='balanced',verbosity=-1,n_jobs=2,random_state=SEED)
 if algo=='xgb':return XGBClassifier(n_estimators=300,max_depth=cfg['depth'],learning_rate=.04,min_child_weight=3,reg_lambda=10,subsample=.85,colsample_bytree=.85,max_bin=63,tree_method='hist',objective='multi:softprob',eval_metric='mlogloss',n_jobs=2,random_state=SEED)
 if algo=='mlp':return MLPClassifier(hidden_layer_sizes=(128,),activation='relu',alpha=cfg['alpha'],learning_rate_init=.001,batch_size=128,max_iter=250,tol=1e-4,n_iter_no_change=15,early_stopping=False,random_state=SEED)
 return ExtraTreesClassifier(n_estimators=250,max_depth=20,min_samples_leaf=3,max_features='sqrt',class_weight='balanced',n_jobs=2,random_state=SEED)

def prepare_view(idx,view):
 ev=[events[i] for i in idx];enc=EvidenceEncoder(*VIEWS[view]).fit(ev)
 rs=enc.transform(ev,training=True);space=Space();a=space.fit(rs);selector=Selector(a,yi[idx],space.names)
 return enc,space,a,selector

def train_new(name,idx,cache):
 cfg=CONFIGS[name];view=cfg['view']
 if view not in cache:cache[view]=prepare_view(idx,view)
 enc,space,a,selector=cache[view];cols=selector.select(cfg['method'],.98,cfg['k'])
 post=ColumnPost(cfg['post']).fit(a[:,cols],space.names[cols]);tx=post.transform(a[:,cols]);target=yi[idx]
 row_weights=np.ones(len(idx),dtype=float)
 if cfg.get('augment',False):
  if '_aug_events' not in cache:
   cache['_aug_events']=parse_events(rare_background(X.iloc[idx],[events[i] for i in idx],SEED+777,count=10))
  aug_rs=enc.transform(cache['_aug_events'],exclude_events=[events[i] for i in idx])
  aug_x=post.transform(space.transform(aug_rs)[:,cols]);tx=csr32(sparse.vstack([tx,aug_x],format='csr'))
  target=np.concatenate([target,target]);row_weights=np.concatenate([row_weights,.5*row_weights])
 est=estimator(cfg)
 if cfg['algo'] in ['xgb','mlp']:est.fit(tx,target,sample_weight=compute_sample_weight('balanced',target)*row_weights)
 elif cfg.get('augment',False):est.fit(tx,target,sample_weight=row_weights)
 else:est.fit(tx,target)
 return Member(name,view,enc,space,cols,post,est)

LEGACY={
 'legacy_lr':dict(k=2048,method='chi2',algo='lr'),
 'legacy_lgb':dict(k=2048,method='f_classif_full',algo='lgb'),
 'legacy_extra':dict(k=1024,method='f_classif_full',algo='extra')}
def train_legacy(idx):
 space=Space();a=space.fit([legacy_records[i] for i in idx]);sel=Selector(a,yi[idx],space.names);result=[]
 for name,cfg in LEGACY.items():
  cols=sel.select(cfg['method'],.95,cfg['k'])
  if cfg['algo']=='lr':est=LogisticRegression(C=.1,class_weight='balanced',max_iter=2000)
  elif cfg['algo']=='lgb':est=LGBMClassifier(n_estimators=250,learning_rate=.05,num_leaves=15,min_child_samples=20,reg_lambda=10,colsample_bytree=.85,class_weight='balanced',verbosity=-1,n_jobs=2,random_state=SEED)
  else:est=ExtraTreesClassifier(n_estimators=250,max_depth=20,min_samples_leaf=3,max_features='sqrt',class_weight='balanced',n_jobs=2,random_state=SEED)
  est.fit(a[:,cols],yi[idx]);post=ColumnPost().fit(a[:,cols],space.names[cols]);result.append(Member(name,'legacy',None,space,cols,post,est))
 return result

def duplicate_frame(frame):
 arr=frame.to_numpy(copy=True)
 for i,row in enumerate(arr):
  for j,v in enumerate(row):
   if isinstance(v,str) and v!='WT':arr[i,j]=v+' '+v
 return pd.DataFrame(arr,columns=frame.columns,index=frame.index)

def rare_background(frame,tr_events,seed,count=20):
 rng=np.random.default_rng(seed);df=Counter((g,t) for row in tr_events for g,ts in row for t in ts)
 syn=[];sub=[]
 for (g,t),n in df.items():
  if n>2:continue
  if kind(t)=='synonymous':syn.append((g,t))
  elif kind(t)=='substitution':sub.append((g,t))
 arr=frame.to_numpy(copy=True);lookup={g:i for i,g in enumerate(frame.columns)}
 for row in arr:
  for _ in range(int(rng.poisson(count))):
   pool=syn if rng.random()<.5 else sub
   g,t=pool[int(rng.integers(len(pool)))];j=lookup[g]
   row[j]=t if not isinstance(row[j],str) or row[j]=='WT' else row[j]+' '+t
 return pd.DataFrame(arr,columns=frame.columns,index=frame.index)

def eval_views(idx,tr_idx,seed):
 clean=X.iloc[idx];noise=rare_background(clean,[events[i] for i in tr_idx],seed);dup=duplicate_frame(clean)
 frames=[clean,noise,dup]
 return frames,[parse_events(f) for f in frames],[list(MutationFeatures(mode='full').dictionaries(f)) for f in frames]

def predict_members(members,weights,frame):
 ev=None;old=None;cache={};p=None
 for m,w in zip(members,weights):
  if m.view=='legacy':
   if old is None:old=list(MutationFeatures(mode='full').dictionaries(frame[genes]))
   rs=old
  else:
   if ev is None:ev=parse_events(frame[genes])
   if m.view not in cache:cache[m.view]=m.encoder.transform(ev)
   rs=cache[m.view]
  q=m.predict_rows(rs);p=q*w if p is None else p+q*w
 return p
print('Novel settings:',len(CONFIGS),'Views:',VIEWS)

# CV-guided targeted refinement: regularization, support threshold, and rare-variant augmentation.
VIEWS['gated_syn_down_s3']=(.25,.25,3)
CONFIGS.update({'raw_mlp_a0.01': {'view': 'raw', 'algo': 'mlp', 'k': 1024, 'alpha': 0.01, 'post': 'maxabs', 'method': 'chi2_full'}, 'raw_mlp_a0.1': {'view': 'raw', 'algo': 'mlp', 'k': 1024, 'alpha': 0.1, 'post': 'maxabs', 'method': 'chi2_full'}, 'gated_syn_down_mlp_a0.01': {'view': 'gated_syn_down', 'algo': 'mlp', 'k': 1024, 'alpha': 0.01, 'post': 'maxabs', 'method': 'chi2_full'}, 'gated_syn_down_mlp_a0.1': {'view': 'gated_syn_down', 'algo': 'mlp', 'k': 1024, 'alpha': 0.1, 'post': 'maxabs', 'method': 'chi2_full'}, 'gated_syn_down_s3_lr_k1024_C0.1': {'view': 'gated_syn_down_s3', 'algo': 'lr', 'k': 1024, 'C': 0.1, 'post': 'none', 'method': 'chi2_full'}, 'gated_syn_down_s3_lr_k1024_C0.5': {'view': 'gated_syn_down_s3', 'algo': 'lr', 'k': 1024, 'C': 0.5, 'post': 'none', 'method': 'chi2_full'}, 'gated_syn_down_s3_lgb_k1024': {'view': 'gated_syn_down_s3', 'algo': 'lgb', 'k': 1024, 'post': 'none', 'method': 'f_classif_full'}, 'gated_syn_down_s3_lr_k2048_C0.1': {'view': 'gated_syn_down_s3', 'algo': 'lr', 'k': 2048, 'C': 0.1, 'post': 'none', 'method': 'chi2_full'}, 'gated_syn_down_s3_lr_k2048_C0.5': {'view': 'gated_syn_down_s3', 'algo': 'lr', 'k': 2048, 'C': 0.5, 'post': 'none', 'method': 'chi2_full'}, 'gated_syn_down_s3_lgb_k2048': {'view': 'gated_syn_down_s3', 'algo': 'lgb', 'k': 2048, 'post': 'none', 'method': 'f_classif_full'}, 'gated_syn_down_s3_xgb_d3': {'view': 'gated_syn_down_s3', 'algo': 'xgb', 'k': 2048, 'depth': 3, 'post': 'none', 'method': 'f_classif_full'}, 'gated_syn_down_s3_xgb_d5': {'view': 'gated_syn_down_s3', 'algo': 'xgb', 'k': 2048, 'depth': 5, 'post': 'none', 'method': 'f_classif_full'}, 'raw_aug_lgb': {'view': 'raw', 'algo': 'lgb', 'k': 2048, 'post': 'none', 'method': 'f_classif_full', 'augment': True}, 'raw_aug_xgb_d5': {'view': 'raw', 'algo': 'xgb', 'k': 2048, 'depth': 5, 'post': 'none', 'method': 'f_classif_full', 'augment': True}, 'gated_s3_aug_lgb': {'view': 'gated_syn_down_s3', 'algo': 'lgb', 'k': 2048, 'post': 'none', 'method': 'f_classif_full', 'augment': True}})
(OUT/'configurations.json').write_text(json.dumps(CONFIGS,indent=2))
print('Total new configurations after refinement:',len(CONFIGS))


Novel settings: 44 Views: {'raw': (1.0, 1.0), 'syn_down': (1.0, 0.25), 'gated': (0.25, 1.0), 'gated_syn_down': (0.25, 0.25), 'gated_nonsyn': (0.25, 0.0)}


## 3. 동일 그룹 CV와 v4 대조군
총 59개 신규 설정과 기존 v4 고정 앙상블을 같은 그룹 3-fold에서 비교합니다. XGBoost/신경망/트리는 선택된 실제 변이 특성을 직접 봅니다.

검증 조건: clean, 학습에서만 고른 희귀 관측 변이 20개 추가(동의적/치환을 절반씩), 동일 토큰 중복 표기. 합성 변이는 실제 정답이 보장된 생물학적 변이가 아니므로 민감도 검사로만 해석합니다. 선택 점수는 **0.6 clean + 0.3 rare-background + 0.1 duplicate**입니다. 새로운 노이즈 정의이므로 이전 버전의 스트레스 수치와 직접 비교하지 않습니다. 테스트 빈도/정답은 사용하지 않습니다.

In [4]:
rows_cv=[];OOF={n:np.zeros((3,len(dev),NC),np.float32) for n in [*CONFIGS,'v4_reference']}
for fold,(a,b) in enumerate(folds):
 tr=dev[a];va=dev[b];cache={};frames,evs,olds=eval_views(va,tr,SEED+901+fold)
 old_members=train_legacy(tr)
 for j,rs in enumerate(olds):OOF['v4_reference'][j,b]=sum(m.predict_rows(rs) for m in old_members)/3
 f=[f1_score(yi[va],OOF['v4_reference'][j,b].argmax(axis=1),average='macro',zero_division=0) for j in range(3)]
 rows_cv.append(dict(model='v4_reference',fold=fold,clean=f[0],noise=f[1],duplicate=f[2],score=.6*f[0]+.3*f[1]+.1*f[2],inputs=5120,seconds=0))
 del old_members
 for name,cfg in CONFIGS.items():
  started=time.time();m=train_new(name,tr,cache);ps=[]
  for j,ev in enumerate(evs):
   p=m.predict_rows(m.encoder.transform(ev));OOF[name][j,b]=p;ps.append(p)
  assert np.allclose(ps[0],ps[2]),'Duplicate annotation invariance failed'
  fs=[f1_score(yi[va],p.argmax(axis=1),average='macro',zero_division=0) for p in ps]
  rows_cv.append(dict(model=name,fold=fold,clean=fs[0],noise=fs[1],duplicate=fs[2],score=.6*fs[0]+.3*fs[1]+.1*fs[2],inputs=len(m.names),seconds=time.time()-started))
  pd.DataFrame(rows_cv).to_csv(OUT/'cv_progress.csv',index=False)
  print(name,'fold',fold,'F1',*[round(v,4) for v in fs],'seconds',round(time.time()-started,1),flush=True)
  del m
cv=pd.DataFrame(rows_cv);cv.to_csv(OUT/'cv_folds.csv',index=False)
ranking=cv.groupby('model').agg(clean=('clean','mean'),noise=('noise','mean'),duplicate=('duplicate','mean'),score=('score','mean'),clean_std=('clean','std'),inputs=('inputs','mean')).sort_values('score',ascending=False)
PLANS={n:[(n,1.)] for n in CONFIGS}
algo_winners=[];seen=set()
for name in ranking.index:
 if name=='v4_reference':continue
 algo=CONFIGS[name]['algo']
 if algo not in seen:algo_winners.append(name);seen.add(algo)
 if len(algo_winners)==4:break
PLANS['new_equal']=[(n,1/len(algo_winners)) for n in algo_winners]
for old_weight in [.25,.5,.75]:
 PLANS[f'mix_v4_{old_weight}']=[('v4_reference',old_weight)]+[(n,(1-old_weight)/len(algo_winners)) for n in algo_winners]
def plan_oof(plan):return sum(OOF[n]*w for n,w in plan)
for name,plan in PLANS.items():
 if name in CONFIGS:continue
 p=plan_oof(plan);fs=[]
 for _,b in folds:fs.append([f1_score(yi[dev[b]],p[j,b].argmax(axis=1),average='macro',zero_division=0) for j in range(3)])
 fs=np.array(fs);ranking.loc[name]=[fs[:,0].mean(),fs[:,1].mean(),fs[:,2].mean(),(.6*fs[:,0]+.3*fs[:,1]+.1*fs[:,2]).mean(),fs[:,0].std(ddof=1),sum(ranking.loc[n,'inputs'] for n,w in plan if w>0)]
ranking=ranking.sort_values('score',ascending=False)
winner=next(n for n in ranking.index if n!='v4_reference');plan=PLANS[winner]
passes_primary=bool(ranking.loc[winner,'score']>=ranking.loc['v4_reference','score']+.002 and ranking.loc[winner,'clean']>=ranking.loc['v4_reference','clean']-.005)
ranking.to_csv(OUT/'model_comparison.csv');np.savez_compressed(OUT/'oof_probabilities.npz',dev=dev,labels=labels,**OOF)
(OUT/'selection.json').write_text(json.dumps({'winner':winner,'plan':plan,'passes_primary_rule':passes_primary,'weights':{'clean':.6,'noise':.3,'duplicate':.1},'algo_winners':algo_winners},indent=2))
print('FROZEN WINNER:',winner,'Plan:',plan,'Primary criterion passed:',passes_primary);display(ranking.head(18))


Evaluating cached complete 59-setting OOF predictions plus paired v4 baseline.
FROZEN WINNER: mix_v4_0.25 Plan: [('v4_reference', 0.25), ('gated_s3_aug_lgb', 0.1875), ('raw_aug_xgb_d5', 0.1875), ('raw_lr_k2048_C0.1', 0.1875), ('gated_syn_down_mlp_a1', 0.1875)] Primary criterion passed: True


,clean,noise,duplicate,score,clean_std,inputs
model,,,,,,
mix_v4_0.25,0.432215,0.339619,0.433093,0.404524,0.002071,12288.0
new_equal,0.425776,0.348769,0.425776,0.402674,0.001932,7168.0
mix_v4_0.5,0.431749,0.321107,0.428534,0.398235,0.006048,12288.0
gated_s3_aug_lgb,0.414637,0.337619,0.414637,0.391532,0.008868,2048.0
mix_v4_0.75,0.426543,0.298013,0.408547,0.386185,0.009986,12288.0
raw_aug_lgb,0.406885,0.323954,0.406885,0.382005,0.006000,2048.0
raw_aug_xgb_d5,0.404134,0.304266,0.404134,0.374174,0.007386,2048.0
raw_lr_k2048_C0.1,0.393797,0.312149,0.393797,0.369302,0.011315,2048.0
v4_reference,0.413232,0.273299,0.384891,0.368418,0.009895,5120.0


## 4. 재분할 민감도 점검
같은 개발 환자를 다른 그룹 분할(seed=57)에서 재평가합니다. 새 환자의 독립 검증은 아닙니다. 이 결과로 가중치나 후보를 다시 조정하지 않고, 선택 후보와 v4만 비교합니다.

In [5]:
def fit_plan(idx,plan,legacy_cache=None):
 cache={};members=[];weights=[];old=legacy_cache
 for name,w in plan:
  if name=='v4_reference':
   if old is None:old=train_legacy(idx)
   members.extend(old);weights.extend([w/3]*3)
  else:members.append(train_new(name,idx,cache));weights.append(w)
 return members,np.array(weights,dtype=float)
confirm=[]
newfolds=list(StratifiedGroupKFold(3,shuffle=True,random_state=57).split(X.iloc[dev],y[dev],groups[dev]))
for fold,(a,b) in enumerate(newfolds):
 tr=dev[a];va=dev[b];frames,_,_=eval_views(va,tr,SEED+1901+fold)
 old=train_legacy(tr);members,ws=fit_plan(tr,plan,legacy_cache=old)
 for title,ms,w in [('v4_reference',old,np.ones(3)/3),('v5_frozen',members,ws)]:
  fs=[]
  for frame in frames:
   p=predict_members(ms,w,frame);fs.append(f1_score(yi[va],p.argmax(axis=1),average='macro',zero_division=0))
  confirm.append(dict(model=title,fold=fold,clean=fs[0],noise=fs[1],duplicate=fs[2],score=.6*fs[0]+.3*fs[1]+.1*fs[2]))
 pd.DataFrame(confirm).to_csv(OUT/'confirmation_cv.csv',index=False)
 print('Confirmation fold',fold,confirm[-2:],flush=True)
 del old,members
confirmation=pd.DataFrame(confirm).groupby('model')[['clean','noise','duplicate','score']].mean();display(confirmation)

passes_confirmation=bool(confirmation.loc['v5_frozen','score']>=confirmation.loc['v4_reference','score'] and confirmation.loc['v5_frozen','clean']>=confirmation.loc['v4_reference','clean']-.005)
print('Confirmation criterion passed:',passes_confirmation)


Confirmation fold 0 [{'model': 'v4_reference', 'fold': 0, 'clean': 0.41368911450421425, 'noise': 0.27624629249893484, 'duplicate': 0.386992118546753, 'score': 0.3697865683068843}, {'model': 'v5_frozen', 'fold': 0, 'clean': 0.4271011635887081, 'noise': 0.324677525880574, 'duplicate': 0.4333004795819449, 'score': 0.39699400387559153}]
Confirmation fold 1 [{'model': 'v4_reference', 'fold': 1, 'clean': 0.4125366085909913, 'noise': 0.26349660791169505, 'duplicate': 0.3724188874090286, 'score': 0.3638128362690061}, {'model': 'v5_frozen', 'fold': 1, 'clean': 0.4320547919591109, 'noise': 0.33813979463312793, 'duplicate': 0.43743215045768297, 'score': 0.40441802861117315}]
Confirmation fold 2 [{'model': 'v4_reference', 'fold': 2, 'clean': 0.40859437852301994, 'noise': 0.26426300698219063, 'duplicate': 0.37597555394299903, 'score': 0.36203308460276906}, {'model': 'v5_frozen', 'fold': 2, 'clean': 0.4255393293508733, 'noise': 0.31681986833514786, 'duplicate': 0.42301892444374994, 'score': 0.392671

/Users/mac/project/cancer_project/.venv/lib/python3.15/site-packages/sklearn/feature_selection/_univariate_selection.py:110: UserWarning: Features [0 0] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/Users/mac/project/cancer_project/.venv/lib/python3.15/site-packages/sklearn/feature_selection/_univariate_selection.py:110: UserWarning: Features [0] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/Users/mac/project/cancer_project/.venv/lib/python3.15/site-packages/sklearn/feature_selection/_univariate_selection.py:110: UserWarning: Features [0] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/Users/mac/project/cancer_project/.venv/lib/python3.15/site-packages/sklearn/feature_selection/_univariate_selection.py:110: UserWarning: Features [0 0] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/Users/m

,clean,noise,duplicate,score
model,,,,
v4_reference,0.411607,0.268002,0.378462,0.365211
v5_frozen,0.428232,0.326546,0.431251,0.398028


## 5. 기존 홀드아웃·전체 학습·저장
선택은 이미 고정했습니다. 기존 1,241명은 여러 번 관측했으므로 새로운 독립 성능이 아닙니다. train 전체로 재학습하고 v1/v3/v4 제출을 보존하며 새 파일을 저장합니다.

In [6]:
hm,hw=fit_plan(dev,plan);hp=predict_members(hm,hw,X.iloc[hold]);pred=labels[hp.argmax(axis=1)]
metrics=dict(macro_f1=f1_score(y[hold],pred,average='macro',zero_division=0),accuracy=accuracy_score(y[hold],pred),weighted_f1=f1_score(y[hold],pred,average='weighted',zero_division=0),n=len(hold))
(OUT/'holdout_metrics.json').write_text(json.dumps(metrics,indent=2))
hreport=pd.DataFrame(classification_report(y[hold],pred,output_dict=True,zero_division=0)).T;hreport.to_csv(OUT/'holdout_classification_report.csv')
pd.DataFrame({'ID':train.ID.iloc[hold].values,'true':y[hold],'prediction':pred}).to_csv(OUT/'holdout_predictions.csv',index=False)
print('Previously observed holdout:',metrics);del hm
models,weights=fit_plan(np.arange(len(train)),plan)
feature_rows=[];sizes=[]
for m,w in zip(models,weights):
 for f in m.raw_names:feature_rows.append({'model':m.name,'view':m.view,'feature':f})
 params=int(sum(z.size for z in m.est.coefs_)+sum(z.size for z in m.est.intercepts_)) if hasattr(m.est,'coefs_') else (int(m.est.coef_.size+m.est.intercept_.size) if hasattr(m.est,'coef_') else None)
 sizes.append({'model':m.name,'view':m.view,'weight':float(w),'inputs':len(m.names),'classifier_parameters':params,'unknown_weight':m.encoder.unknown if m.encoder else None,'silent_weight':m.encoder.silent if m.encoder else None,'support':m.encoder.support if m.encoder else None,'training_iterations':np.asarray(m.est.n_iter_).tolist() if hasattr(m.est,'n_iter_') else None})
features=pd.DataFrame(feature_rows);features.to_csv(OUT/'final_features.csv',index=False)
(OUT/'model_sizes.json').write_text(json.dumps(sizes,indent=2))
def predict_bundle(bundle,frame):return predict_members(bundle['models'],bundle['weights'],frame)
bundle={'version':5,'models':models,'weights':weights,'genes':genes,'classes':labels,'winner':winner,'plan':plan,'holdout_metrics':metrics,'sizes':sizes,'predict':predict_bundle}
model_path=OUT/'cancer_model_v5.pkl'
with model_path.open('wb') as f:cloudpickle.dump(bundle,f,protocol=pickle.HIGHEST_PROTOCOL)
with model_path.open('rb') as f:loaded=pickle.load(f)
tp=loaded['predict'](loaded,test)
assert tp.shape==(len(test),NC) and np.isfinite(tp).all() and np.allclose(tp.sum(axis=1),1)
submission=sample.copy();submission.SUBCLASS=labels[tp.argmax(axis=1)]
assert submission.ID.equals(sample.ID) and submission.SUBCLASS.isin(labels).all()
submission.to_csv(OUT/'submission_v5.csv',index=False);submission.to_csv(SUBMIT/'혜진_submission_v5.csv',index=False)
pd.DataFrame(tp,index=test.ID,columns=labels).to_csv(OUT/'test_probabilities.csv')
print('MODEL:',model_path.resolve(),'MiB:',round(model_path.stat().st_size/1024**2,2));display(FileLink(str(model_path)))
print('SUBMISSION:',(SUBMIT/'혜진_submission_v5.csv').resolve());display(pd.DataFrame(sizes))


Previously observed holdout: {'macro_f1': 0.4667540950492095, 'accuracy': 0.45850120870265915, 'weighted_f1': 0.4632297154760839, 'n': 1241}
MODEL: /Users/mac/project/cancer_project/artifacts/v5/cancer_model_v5.pkl MiB: 68.85
SUBMISSION: /Users/mac/project/cancer_project/submission/혜진_submission_v5.csv


/Users/mac/project/cancer_project/.venv/lib/python3.15/site-packages/sklearn/feature_selection/_univariate_selection.py:110: UserWarning: Features [0 0] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/Users/mac/project/cancer_project/.venv/lib/python3.15/site-packages/sklearn/feature_selection/_univariate_selection.py:110: UserWarning: Features [0] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/Users/mac/project/cancer_project/.venv/lib/python3.15/site-packages/sklearn/feature_selection/_univariate_selection.py:110: UserWarning: Features [0] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/Users/mac/project/cancer_project/.venv/lib/python3.15/site-packages/sklearn/feature_selection/_univariate_selection.py:110: UserWarning: Features [0 0] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/Users/m

/Users/mac/project/cancer_project/artifacts/v5/cancer_model_v5.pkl

,model,view,weight,inputs,classifier_parameters,unknown_weight,silent_weight,support,training_iterations
0,legacy_lr,legacy,0.083333,2048,53274.0,NaN,NaN,NaN,[34]
1,legacy_lgb,legacy,0.083333,2048,NaN,NaN,NaN,NaN,250
2,legacy_extra,legacy,0.083333,1024,NaN,NaN,NaN,NaN,None
3,gated_s3_aug_lgb,gated_syn_down_s3,0.187500,2048,NaN,0.25,0.25,3.0,300
4,raw_aug_xgb_d5,raw,0.187500,2048,NaN,1.00,1.00,2.0,None
5,raw_lr_k2048_C0.1,raw,0.187500,2048,53274.0,1.00,1.00,2.0,[90]
6,gated_syn_down_mlp_a1,gated_syn_down,0.187500,1024,134554.0,0.25,0.25,2.0,114


## 6. 결과 문서와 독립 재로딩 검증
새 대회 점수는 제출 후 확인합니다. 전체 테스트를 새 로딩 모델로 예측해 CSV·확률·ID 일치를 확인하고, 프로젝트 Python 파일 없는 폴더에서 추론합니다. 실제 선택된 기법만 최종 적용이라고 기록합니다.

In [7]:
import subprocess,tempfile,importlib.metadata
program="""
import pickle,sys,json,numpy as np,pandas as pd
with open(sys.argv[1],'rb') as f:b=pickle.load(f)
d=pd.read_csv(sys.argv[2],nrows=4);p=b['predict'](b,d)
assert np.isfinite(p).all() and np.allclose(p.sum(axis=1),1)
assert not {'mutation_model','report_writer'}.intersection(sys.modules)
print(json.dumps({'status':'passed','version':b['version'],'rows':len(d)}))
"""
env=os.environ.copy();env.pop('PYTHONPATH',None)
with tempfile.TemporaryDirectory() as td:
 check=subprocess.run([sys.executable,'-c',program,str(model_path.resolve()),str((DATA/'test.csv').resolve())],cwd=td,env=env,text=True,capture_output=True,check=True)
independent=json.loads(check.stdout)
p2=loaded['predict'](loaded,test)
assert np.allclose(p2,tp) and np.array_equal(labels[p2.argmax(axis=1)],submission.SUBCLASS.to_numpy())
assert not set(groups[dev]).intersection(groups[hold])
versions={p:importlib.metadata.version(p) for p in ['numpy','pandas','scipy','scikit-learn','lightgbm','xgboost','matplotlib','cloudpickle','threadpoolctl','ipython']}
(WORK/'requirements-v5.txt').write_text('\n'.join(p+'=='+v for p,v in versions.items())+'\n')
meta={'status':'passed','version':5,'new_public_score':None,'v4_public_score':0.3169555443,'winner':winner,'plan':plan,'sizes':sizes,'holdout_metrics':metrics,'primary_criterion_passed':passes_primary,'confirmation_criterion_passed':passes_confirmation,'confirmation':confirmation.to_dict('index'),'independent_loading':independent,'packages':versions,'sha256':{p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in [DATA/'train.csv',DATA/'test.csv',model_path,SUBMIT/'혜진_submission_v5.csv']}}
(OUT/'verification.json').write_text(json.dumps(meta,indent=2))
def table5(df):
 return '| '+' | '.join(map(str,df.columns))+' |\n| '+' | '.join(['---']*len(df.columns))+' |\n'+'\n'.join('| '+' | '.join(f'{v:.5f}' if isinstance(v,float) else str(v) for v in row)+' |' for row in df.itertuples(index=False,name=None))
text=f"""# 혜진 v5 비교·검증 보고서

## 결론과 한계
v4 실제 Public은 **0.3169555443**. v5 Public은 **미측정**입니다. 새 성능 향상을 보장하지 않습니다. 후보는 {winner}, 구성은 {plan}입니다. 1차 CV의 채택 기준 통과 여부: {passes_primary}. 재분할 기준 통과 여부: {passes_confirmation}. 어느 한 기준이라도 미통과라면 기존 v4보다 검증된 개선이라고 해석하지 마세요.

기존 반복 관측 홀드아웃: {json.dumps(metrics,ensure_ascii=False)}. v4의 같은 홀드아웃 Macro F1은 0.4445441693이었으나 CV와 과거 홀드아웃 모두 반복 활용되어 새로운 환자의 독립 검증이 아닙니다.

## 실제 모델 구성
{table5(pd.DataFrame(sizes))}

모델은 실제 변이 특성을 직접 학습했습니다. 일부 앙상블은 v4의 고정된 3개 모델을 재학습한 기준과 새 모델을 섞습니다. 기존 모델 파일이 없어도 노트북에서 재학습할 수 있습니다. 가중치는 개발 OOF 비교로 고정했고 테스트 암종 비율에 강제로 맞추지 않았습니다.

## 파서와 파생변수
* X→* 종결 표기, p. 접두사, K16Rfs/K16fs 형태의 프레임시프트 표기를 통일합니다. 유전체/전사체 좌표의 완전한 정규화는 아닙니다.
* 같은 유전자 셀 안의 동일 토큰을 제거합니다. 개별 변이의 기능이 같다는 가정으로 다른 토큰을 모두 지우지 않습니다.
* aa:G:R>H 같은 위치 독립 아미노산 변화 단서를 추가합니다. 서로 다른 위치의 효과가 같다는 보장은 없으며, 기존 exact/position 단서를 함께 유지합니다.
* gene/variant/position/type의 값은 학습 지원도에 따른 가중치일 수 있습니다. 구조적 표기(stop/frameshift/deletion/insertion)는 1, 알려진 치환은 1, 같은 AA변화가 알려졌으면 최대 0.75, 낯선 치환은 0.25 또는 대조군 1을 비교합니다. 동의적 변이는 0/0.25/1을 비교했습니다. 병원성이나 무해함을 확정하는 규칙이 아닙니다.
* 지원도는 해당 학습 환자에서만 적합하고, 학습 행은 자기 출현 1을 제외합니다. 검증/테스트 빈도를 적합하지 않습니다. 사전은 후보별 관측 환자 수 2명 또는 3명 이상인 토큰/AA변화를 저장합니다.
* raw 유전자 수, 가중 유전자 수, 비동의적/절단 형태 유전자 수, 유전자별 유형 비율을 요약합니다. 토큰 개수·평균 위치를 환자 전체 핵심 요약으로 쓰지 않습니다. raw 변이 유전자 수를 보존해 행 정규화로 변이량 정보가 모두 사라지지 않도록 했습니다. 임상 TMB나 원인 변이 판정은 아닙니다.
* 7개 수작업 유전자 그룹은 해당 유전자 가중치의 합을 log1p로 집계합니다. 경로 활성도라고 해석하지 않습니다.

## 비교와 선택
초기 44개 설정에 개발 CV 기반 규제/지원도 12개·증강 3개를 더한 신규 단일 설정 {len(CONFIGS)}개와 v4 기준 앙상블을 동일 그룹 3-fold에서 비교했습니다. min_df=3, 타깃 연관성·상관 중복 제거·지원도·TF-IDF/스케일러는 학습 부분에서 적합합니다. 주로 1,024/2,048개 입력을 유지하고 상관 임계값은 새 후보 0.98, v4 기준 0.95입니다.

LR, LightGBM, XGBoost, MLP, ExtraTrees를 비교했습니다. MLP는 학습 클래스 균형 가중치를 사용하고 내부 무작위 validation 대신 고정 최대 250 epoch와 학습 loss 정지 규칙을 사용했습니다. 최대 epoch 도달은 수렴을 보장하지 않습니다. block LR은 변이 단서만 TF-IDF/L2 정규화하고 환자 요약을 별도로 보존합니다. 전역 벡터 L2 정규화와 다릅니다. 최종 포함 여부는 위 실제 구성을 보세요.

선택 지표는 0.6 clean + 0.3 희귀 배경 변이 + 0.1 중복 표기입니다. 기준 채택 조건은 robust score가 v4보다 0.002 이상 높고 clean F1이 v4 대비 0.005를 넘게 떨어지지 않는 것입니다. 이 기준은 대회 점수를 보장하지 않습니다. 같은 개발 데이터에서 여러 후보를 고르므로 선택 편향이 있습니다.

{table5(ranking.reset_index())}

## 스트레스 정의와 현실성 한계
배경 변이는 학습 환자 2명 이하에서 관측된 실제 동의적/치환 토큰을 절반씩 무작위로 골라 평균 20개 추가합니다. 실제 희귀 변이가 모두 무해하거나 라벨을 유지한다는 보장은 없습니다. label-independent distractor 민감도 실험이지 실제 Test와 같은 분포가 아닙니다. 특정 후보가 이 합성 조건에 잘 맞았다고 실제 임상/대회 성능을 확정하지 않습니다.
중복 조건은 같은 토큰을 복사해 표현 개수만 늘린 것입니다. v5는 중복 제거 때문에 예측 불변성을 검사합니다. 실제 다중 변이나 전사체 차이를 완전히 재현한 것은 아닙니다. 테스트 통계로 노이즈 확률을 적합하거나 테스트의 일반인/정상 변이를 확정해 삭제하지 않았습니다.

## 희귀 변이 증강 설정
추가 3개 후보에서는 실제 학습에서만 추출한 희귀 변이를 평균 10개 추가한 복사본을 만들고 0.5 가중치로 학습했습니다. 원본과 증강본의 라벨은 같다는 가정입니다. 사전/지원도/선별은 원본 학습으로만 적합했고 증강본은 사전 빈도를 늘리지 않습니다. 자기 출현 제외는 원래 행에 있던 토큰/AA변화에만 적용했습니다. 실제 생물학적 변이가 라벨을 바꾸지 않는다는 보장은 없으며, 이 합성 가정에 잘 맞는 모델이 선택될 위험이 있습니다.

## 다른 그룹 분할 재점검
같은 개발 환자에 seed=57의 3-fold를 적용한 민감도 점검입니다. 새 독립 환자 데이터가 아니며 이 결과로 모델/가중치를 다시 튜닝하지 않았습니다.

{table5(confirmation.reset_index())}

## 기존 홀드아웃 클래스별 결과
{table5(hreport.reset_index())}

## 변수 수와 저장 파일
서로 다른 view는 같은 이름이어도 가중치·해석이 다를 수 있습니다. final_features.csv에서 view와 feature를 함께 보세요. 단순 이름 합집합을 v4의 2,063개와 곧바로 비교하면 오해가 생길 수 있습니다.
피클 크기는 {model_path.stat().st_size/1024**2:.2f} MiB, 노트북 Run All로 학습부터 저장까지 실행됩니다. 새 제출은 혜진_submission_v5.csv이며 기존 파일을 덮어쓰지 않았습니다. 피클은 전처리 코드를 포함하고 bundle['predict'](bundle, frame)으로 추론합니다. 독립 폴더 로딩/예측 및 전체 테스트 CSV와 확률·ID 일치를 확인했습니다. 환경은 requirements-v5.txt, 검증/해시는 artifacts/v5/verification.json입니다.

## 점수 이력
v1 Public 0.2765425197 → v3 0.2998780222 → v4 0.3169555443. v5는 미측정입니다. 기존 알려진 점수를 새 모델 성능으로 표시하지 않습니다. 외부 데이터, 테스트 적합, 의사 라벨, 정답 복사, 테스트 비율 강제 정렬은 사용하지 않았습니다.
"""
(WORK/'analysis_report_v5.md').write_text(text)
print('Verified:',meta['status'],'Report:',WORK/'analysis_report_v5.md','Holdout:',metrics)


Verified: passed Report: /Users/mac/project/cancer_project/analysis_report_v5.md Holdout: {'macro_f1': 0.4667540950492095, 'accuracy': 0.45850120870265915, 'weighted_f1': 0.4632297154760839, 'n': 1241}


## 7. 전체 테스트 재로딩·변수 사전·최종 보고서
배치 독립성, CSV/확률/ID 정합성, 실제 입력 정의 수를 검증합니다. v5는 용량과 관점이 늘어난 후보이며 새 Public 점수는 별도 확인해야 합니다.

In [8]:
# Final v5 audit and portable reports. No previous model files are required.
from pathlib import Path
from collections import Counter
import json,pickle,hashlib,sys
import numpy as np
import pandas as pd
OUT=WORK/'artifacts'/'v5'
selection=json.loads((OUT/'selection.json').read_text())
ranked=pd.read_csv(OUT/'model_comparison.csv',index_col=0)
# Correct a diagnostic width: nonzero ensemble components all require their input slots.
config=json.loads((OUT/'configurations.json').read_text());base_names=json.loads((OUT/'selection.json').read_text())['algo_winners']
ranked.loc['new_equal','inputs']=sum(ranked.loc[n,'inputs'] for n in base_names)
for weight in [.25,.5,.75]:ranked.loc[f'mix_v4_{weight}','inputs']=ranked.loc['v4_reference','inputs']+ranked.loc['new_equal','inputs']
ranked.to_csv(OUT/'model_comparison.csv')
with (OUT/'cancer_model_v5.pkl').open('rb') as f:final=pickle.load(f)
probs=final['predict'](final,test)
saved=pd.read_csv(SUBMIT/'혜진_submission_v5.csv');stored=pd.read_csv(OUT/'test_probabilities.csv',index_col=0)
assert saved.ID.equals(sample.ID) and saved.ID.equals(test.ID) and list(saved)==list(sample)
assert not saved.isna().any().any() and np.isfinite(probs).all() and np.allclose(probs.sum(axis=1),1)
assert list(stored.columns)==list(final['classes']) and list(stored.index)==list(test.ID)
assert np.allclose(probs,stored.to_numpy()) and np.array_equal(saved.SUBCLASS.to_numpy(),final['classes'][probs.argmax(axis=1)])
assert np.allclose(probs[:8],final['predict'](final,test.iloc[:8]))
manifest=pd.read_csv(OUT/'split_manifest.csv',dtype={'group':str});assert manifest.groupby('group')['split'].nunique().max()==1
meta=json.loads((OUT/'verification.json').read_text())
meta.update(full_independent_test_reload='passed',batch_invariance='passed',profile_split_overlap=0,python=sys.version,configurations=len(config))
features=pd.read_csv(OUT/'final_features.csv')
meta['input_slots']=int(len(features));meta['distinct_view_feature_definitions']=int(len(features[['view','feature']].drop_duplicates()))
meta['model_bytes']=(OUT/'cancer_model_v5.pkl').stat().st_size
(OUT/'verification.json').write_text(json.dumps(meta,indent=2))
# Full original-column dictionary, without inventing biological functional annotations.
tr_mask=train[genes].notna()&train[genes].ne('WT');te_mask=test[genes].notna()&test[genes].ne('WT')
uses=Counter();views={g:set() for g in genes}
for row in features.itertuples(index=False):
 parts=row.feature.split(':')
 if parts[0] in {'gene','type','variant','position','aa','supported'} and parts[1] in views:
  uses[parts[1]]+=1;views[parts[1]].add(row.view)
gd=pd.DataFrame({'gene':genes,'train_mutated':tr_mask.sum().values,'train_rate':tr_mask.mean().values,'test_rate':te_mask.mean().values,'selected_model_feature_uses':[uses[g] for g in genes],'selected_views':[', '.join(sorted(views[g])) or '-' for g in genes]})
gd.to_csv(OUT/'gene_dictionary_v5.csv',index=False)
def _table(df):
 return '| '+' | '.join(map(str,df.columns))+' |\n| '+' | '.join(['---']*len(df.columns))+' |\n'+'\n'.join('| '+' | '.join(f'{v:.5f}' if isinstance(v,float) else str(v) for v in r)+' |' for r in df.itertuples(index=False,name=None))
(WORK/'gene_dictionary_v5.md').write_text('# v5 전체 유전자 컬럼 사전\n\nWT/변이 표기 컬럼의 관측 통계입니다. 유전자 기능 주석은 원본에 없어 추정하지 않습니다. selected_model_feature_uses는 모델 간 중복을 포함한 gene/type/variant/position/aa/supported 연결 변수 수입니다. 0이어도 유전자 전체 요약이나 module에 기여할 수 있습니다. view는 같은 이름이라도 가중치와 전처리 의미가 다를 수 있습니다. test_rate는 기술통계이며 학습·선별·확률 보정에 사용하지 않았습니다.\n\n'+_table(gd))
view_features=features.groupby(['view','feature'],sort=True).model.agg(lambda z:', '.join(z)).reset_index()
(WORK/'selected_variables_v5.md').write_text('# v5 선택 변수 사전\n\n총 분류기 입력 슬롯은 '+str(len(features))+'개, view와 feature 쌍 기준은 '+str(len(view_features))+'개입니다. 서로 다른 view의 같은 이름을 같은 숫자 특성으로 간주하지 않습니다. AA 변화는 위치를 무시한 보완 단서이며 기능적 동일성 판정이 아닙니다. 가중치/지원도 설정은 model_sizes.json, 전체 파생변수 의미는 analysis_report_v5.md를 참고하세요.\n\n'+_table(view_features))
# Compare both recorded CV splits, not just the favorable holdout.
conf=pd.read_csv(OUT/'confirmation_cv.csv').groupby('model')[['clean','noise','duplicate','score']].mean()
primary=ranked.loc[[ 'v4_reference',selection['winner']],['clean','noise','duplicate','score']].copy();primary.index=['v4','v5']
confirm=conf.loc[['v4_reference','v5_frozen']].copy();confirm.index=['v4','v5']
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
fig,axes=plt.subplots(1,2,figsize=(12,4.5))
for ax,df,title in [(axes[0],primary,'Development group CV'),(axes[1],confirm,'Different group split (same patients)')]:
 x=np.arange(3)
 for i,(name,row) in enumerate(df.iterrows()):ax.bar(x+(i-.5)*.35,row[['clean','noise','duplicate']].to_numpy(float),width=.35,label=name)
 ax.set_xticks(x,['Clean','Rare background','Duplicate']);ax.set_ylim(0,.55);ax.set_ylabel('Macro F1');ax.set_title(title);ax.legend()
fig.tight_layout();fig.savefig(OUT/'validation_comparison_v5.png',dpi=150);plt.close(fig)
notes=f"""

## 최종 검증·실제 입력 수
전체 테스트 2,546명을 독립 재로딩 모델로 예측해 저장된 확률·CSV와 일치함을 확인했습니다. 8명만 별도 추론해도 같은 확률이 나와 배치 내 다른 환자의 빈도로 확률을 보정하지 않음을 점검했습니다. train 내부 그룹 분리 검사를 통과했습니다. 이 검증은 대회 정확도를 알려주지 않습니다.

분류기별 입력 슬롯 합계는 **{len(features):,}개**, view/feature 쌍 기준 서로 다른 정의는 **{len(view_features):,}개**입니다. v4의 단일 표현 합집합 2,063개와 단순 비교하지 마세요. v5는 추가 관점·모델을 결합하여 용량과 추론 비용이 늘었습니다. 피클은 **{(OUT/'cancer_model_v5.pkl').stat().st_size/1024**2:.2f} MiB**로 v4 약 40.28 MiB보다 큽니다. 시험한 59개 설정 전체를 배포하는 것은 아니며 아래 구성만 사용합니다.

{_table(pd.DataFrame(meta['sizes']))}

표의 지원도는 unknown_weight가 1보다 작을 때의 가중 표현에서 사용합니다. raw 표현은 지원도 사전을 이용한 변이 감쇠를 사용하지 않습니다. 기존 v4 기준은 원래 설정으로 재학습했습니다. 분류기 모두 선택된 실제 변이 특성을 직접 학습하며, 로지스틱 점수 26개만을 트리에 입력한 스태킹 모델이 아닙니다.

### 같은 조건의 기준 비교
{_table(primary.reset_index(names='model'))}

### 재분할 점검
{_table(confirm.reset_index(names='model'))}

두 재분할은 같은 환자들로 구성되어 새로운 외부 검증이 아닙니다. 기존 홀드아웃 재평가 F1은 **{meta['holdout_metrics']['macro_f1']:.5f}**, 정확도는 **{meta['holdout_metrics']['accuracy']:.5f}**입니다. v4의 해당 값은 0.4445441693과 0.4375503626이었습니다. v5 Public은 아직 미측정입니다. 반복 개발·후보 선택 편향과 합성 스트레스의 현실성 한계가 남아 있습니다.

![동일 조건의 v4-v5 검증 비교](artifacts/v5/validation_comparison_v5.png)

최종 앙상블에는 v4 기준이 일부 포함되어, 새로운 표현의 개별 중복 불변성이 전체 앙상블의 완전 불변성을 뜻하지 않습니다. 그림은 3-fold 평균이며 신뢰구간이나 새로운 환자의 성능 보장이 아닙니다. 선택 변수 사전은 selected_variables_v5.md, 전체 원본 유전자 통계는 gene_dictionary_v5.md입니다. 사용 환경은 requirements-v5.txt와 verification.json에 기록되어 있습니다.
"""
# Keep the displayed input-width diagnostic consistent with the actual slot counts.
report=WORK/'analysis_report_v5.md'
text=report.read_text()
start=text.index('| model | clean | noise | duplicate | score | clean_std | inputs |')
end=text.index('\n## 스트레스 정의',start)
text=text[:start]+_table(ranked.reset_index())+'\n'+text[end:]
marker='\n\n## 최종 검증·실제 입력 수\n'
report.write_text(text.split(marker)[0]+notes)
if meta['primary_criterion_passed'] and meta['confirmation_criterion_passed']:
 (WORK/'analysis_report.md').write_text(report.read_text())
 (WORK/'gene_dictionary.md').write_text((WORK/'gene_dictionary_v5.md').read_text())
print('V5 final checks passed:',meta['holdout_metrics'])
print('Input slots:',meta['input_slots'],'Distinct view/feature definitions:',meta['distinct_view_feature_definitions'])
print('Public score: not measured. Submission:',(SUBMIT/'혜진_submission_v5.csv').resolve())


V5 final checks passed: {'macro_f1': 0.4667540950492095, 'accuracy': 0.45850120870265915, 'weighted_f1': 0.4632297154760839, 'n': 1241}
Input slots: 12288 Distinct view/feature definitions: 7193
Public score: not measured. Submission: /Users/mac/project/cancer_project/submission/혜진_submission_v5.csv


Matplotlib is building the font cache; this may take a moment.
